In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
from transformers import VoxtralProcessor

model_name_or_path = "/projects/bhuang/models/llm/pretrained/mistralai/Voxtral-Mini-3B-2507"
# model_name_or_path = "/projects/bhuang/models/llm/pretrained/mistralai/Voxtral-Small-24B-2507"
# model_name_or_path = "/home/bhuang/llm/trl/outputs/intent_classification/audio_ft/sga/sft_voxtral_mini_3b_2507_lora_r64_ep40_bs64_lr2e4_merged"
# model_name_or_path = "/home/bhuang/llm/trl/outputs/summary/audio_ft/edenred/sft_voxtral_mini_3b_2507_lora_r64_ep3_bs128_lr1e4_merged"
# model_name_or_path = "/home/bhuang/llm/trl/outputs/asr/audio_ft/zaion/sft_voxtral_mini_3b_2507_lora_r64_ep3_bs32_lr1e4_merged"

processor = VoxtralProcessor.from_pretrained(model_name_or_path)

In [ ]:
import torch
from transformers import VoxtralForConditionalGeneration

model = VoxtralForConditionalGeneration.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
).eval()

type(model)

In [ ]:
# tmp
sys_promt_file = "/home/bhuang/llm/momo/summary/prompts/summarize_audio_free_text.txt"
with open(sys_promt_file, "r") as f:
    sys_prompt = f.read()

print(sys_prompt)

In [ ]:
# audio_filepath = "/home/bhuang/llm/momo/intent_classification/data/sga/audio_degraded/000000.wav"
# audio_filepath = "/home/bhuang/llm/momo/intent_classification/data/databank/audio_degraded/train/000000.wav"
# audio_filepath = "/projects/bhuang/corpus/text/summary/edenred/processed/audios/20241223_20241229/c8e3b9ac-96a8-4d2f-a284-ad2586639567/c8e3b9ac-96a8-4d2f-a284-ad2586639567.wav"
# audio_filepath = "/projects/bhuang/corpus/speech/zaion/segmented_hm_hm_nemo_16k/190h/train/concat/728b1cae-eead-40c6-b2ad-b0ebe7227c22.wav"
audio_filepath = "/projects/bhuang/corpus/speech/zaion/segmented_hm_hm_nemo_16k/190h/test/2021_05_28/00049_4.wav"

messages = [
    # {
    #     "role": "system",
    #     "content": [
    #         {
    #             "type": "text",
    #             "text": "You are an assistant that transcribes speech accurately.",
    #         }
    #     ],
    # },
    {
        "role": "user",
        "content": [
            # voxtral
            # {"type": "audio", "url": audio_url},
            {"type": "audio", "path": audio_filepath},
            # {"type": "audio", "base64": audio_base64},
            # gemma-3n ?
            # {"type": "audio", "audio": audio_filepath},
            # {"type": "text", "text": "Please transcribe this audio."},
            {"type": "text", "text": "Transcribe the audio exactly as spoken."},
            # {"type": "text", "text": sys_prompt},
        ],
    },
    # {
    #     "role": "assistant",
    #     "content": [
    #         {"type": "text", "text": "Hello! How are you?"},
    #     ],
    # },
]

inputs = processor.apply_chat_template(
    # messages,
    messages,
    # add_generation_prompt=True,
    # tokenize=False,
    # return_dict=True,
    # return_tensors="pt",
    # continue_final_message=True,
    # padding=True,
)

# inputs = processor.apply_transcription_request(
#     language="fr",
#     audio=audio_filepath,
#     model_id=model_name_or_path,
#     tokenize=False,
# )

list(inputs.keys()), inputs

In [ ]:
with torch.inference_mode():
    outputs = model.generate(
        **inputs.to(model.device, dtype=model.dtype),
        max_new_tokens=1024,
        # do_sample=False,
        # disable_compile=True,
        temperature=0.2,
        do_sample=True,
        top_p=0.95,
    )

decoded_outputs = processor.batch_decode(
    outputs[:, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
    # clean_up_tokenization_spaces=True,
)

print(decoded_outputs[0])